In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pulp as plp

In [ ]:
# Initial conditions
CAPACITY = 35.0  # kWh
INITIAL_STATE_OF_CHARGE = 20.0  # kWh

In [ ]:
# INPUTS
forecast_horizon = 48
period = 24
forecast_timesteps = np.arange(forecast_horizon)

price_forecast = 3 * np.sin(2 * np.pi * forecast_timesteps / period) + 5
price_forecast = np.maximum(price_forecast + np.random.normal(size=(forecast_horizon,)), 0.01)

demand_forecast = 4 * np.sin(2 * np.pi * (forecast_timesteps) / period) + 6

plt.plot(price_forecast, label="Price (£ / kWh)", drawstyle="steps-post", marker="o")
plt.plot(demand_forecast, label="Demand (kWh)", drawstyle="steps-post", marker="x")
plt.legend()

In [ ]:
# Greedy solver

def greedy(
    state_of_charge: float,
    demand: float,
    price: float,
    capacity: float = CAPACITY
) -> float:

    problem = plp.LpProblem(name="battery_charging", sense=plp.LpMinimize)

    # Decision variable
    grid_inflow = plp.LpVariable(name="grid_inflow", lowBound=0, cat="Continuous")

    # Objective function
    problem += grid_inflow * price

    # Constraints
    problem += grid_inflow >= demand - state_of_charge
    problem += grid_inflow <= capacity - state_of_charge + demand

    problem.solve(plp.PULP_CBC_CMD(msg=False))
    return grid_inflow.varValue


def mpc(
    initial_state_of_charge: float,
    demand_forecast: np.ndarray,
    price_forecast: np.ndarray,
    capacity: float = CAPACITY,
    horizon: int = 10,
    discount_factor: float = 0.95,
) -> float:
    
    assert demand_forecast.ndim == 1
    assert price_forecast.ndim == 1
    assert horizon <= len(demand_forecast)
    assert horizon <= len(price_forecast)
    
    timesteps = range(horizon)

    problem = plp.LpProblem(name="battery_charging", sense=plp.LpMinimize)

    # Decision variables
    grid_inflow = plp.LpVariable.dict(name="grid_inflow", indices=(timesteps, ), lowBound=0, cat="Continuous")
    state_of_charge = plp.LpVariable.dict(name="state_of_charge", indices=(timesteps, ), lowBound=0, cat="Continuous")

    # Objective function
    problem += plp.lpSum(
        [
            (discount_factor ** k) * grid_inflow[k] * price_forecast[k]
            for k in timesteps
        ]
    )

    # Constraints
    
    # total supply has to meet demand
    for k in timesteps:
        problem += grid_inflow[k] >= demand_forecast[k] - state_of_charge[k]

    # cannot charge beyond capacity
    for k in timesteps:
        problem += grid_inflow[k] <= capacity - state_of_charge[k] + demand_forecast[k]

    # transition constraint
    for k in timesteps[:-1]:
        problem += state_of_charge[k+1] == state_of_charge[k] - demand_forecast[k] + grid_inflow[k]

    # inital constraint
    problem += state_of_charge[0] == initial_state_of_charge
    
    problem.solve(plp.PULP_CBC_CMD(msg=False))
    return grid_inflow[0].varValue


In [ ]:
def transition(
    current_state_of_charge: float,
    grid_flow: float,
    demand_forecast: np.ndarray,
    price_forecast: np.ndarray,
):
    current_demand = demand_forecast[0]
    new_demand_forecast = demand_forecast[1:]
    current_price = price_forecast[0]
    new_price_forecast = price_forecast[1:]

    # Update state given current state and action
    new_state_of_charge = current_state_of_charge - current_demand + grid_flow

    # Compute total cost of charge
    cost_of_charge = grid_flow * current_price

    return (
        new_state_of_charge,
        new_demand_forecast,
        new_price_forecast,
        cost_of_charge
    )

In [ ]:
def simulate_greedy(
    demand_forecast: np.ndarray,
    price_forecast: np.ndarray,
    n_timesteps: int,
    initial_state_of_charge: float = INITIAL_STATE_OF_CHARGE,
    capacity: float = CAPACITY,
):
    
    grid_flow_history = []
    state_of_charge_history = [initial_state_of_charge]
    cost_history = []

    current_state_of_charge = initial_state_of_charge
    
    for t in range(n_timesteps):
        # Take action given current state
        grid_flow = greedy(
            state_of_charge=current_state_of_charge,
            demand=demand_forecast[0],
            price=price_forecast[0],
            capacity=capacity
        )
        grid_flow_history.append(grid_flow)

        # Transition to new state given action
        (
            current_state_of_charge,
            demand_forecast,
            price_forecast,
            cost_of_charge,
        ) = transition(
            current_state_of_charge=current_state_of_charge,
            grid_flow=grid_flow,
            demand_forecast=demand_forecast,
            price_forecast=price_forecast
        )
        state_of_charge_history.append(current_state_of_charge)
        cost_history.append(cost_of_charge)

    return (
        grid_flow_history,
        state_of_charge_history,
        cost_history,
    )

In [ ]:
def simulate_mpc(
    demand_forecast: np.ndarray,
    price_forecast: np.ndarray,
    n_timesteps: int,
    initial_state_of_charge: float = INITIAL_STATE_OF_CHARGE,
    capacity: float = CAPACITY,
    horizon: int = 10,
    discount_factor: float = 0.95
):
    assert n_timesteps + horizon <= len(demand_forecast)
    assert n_timesteps + horizon <= len(price_forecast)
    
    grid_flow_history = []
    state_of_charge_history = [initial_state_of_charge]
    cost_history = []

    current_state_of_charge = initial_state_of_charge
    
    for t in range(n_timesteps):
        # Take action given current state
        grid_flow = mpc(
            initial_state_of_charge=current_state_of_charge,
            demand_forecast=demand_forecast,
            price_forecast=price_forecast,
            capacity=capacity,
            horizon=horizon,
            discount_factor=discount_factor,
        )
        grid_flow_history.append(grid_flow)

        # Transition to new state given action
        (
            current_state_of_charge,
            demand_forecast,
            price_forecast,
            cost_of_charge,
        ) = transition(
            current_state_of_charge=current_state_of_charge,
            grid_flow=grid_flow,
            demand_forecast=demand_forecast,
            price_forecast=price_forecast
        )
        state_of_charge_history.append(current_state_of_charge)
        cost_history.append(cost_of_charge)

    return (
        grid_flow_history,
        state_of_charge_history,
        cost_history,
    )

In [ ]:
simulation_timesteps = 30

(
    grid_flow_history_greedy,
    state_of_charge_history_greedy,
    cost_history_greedy,
) = simulate_greedy(
    demand_forecast=demand_forecast,
    price_forecast=price_forecast,
    n_timesteps=simulation_timesteps,
)

(
    grid_flow_history_mpc,
    state_of_charge_history_mpc,
    cost_history_mpc,
) = simulate_mpc(
    demand_forecast=demand_forecast,
    price_forecast=price_forecast,
    n_timesteps=simulation_timesteps,
)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)

axes[0].plot(demand_forecast[:simulation_timesteps], label="Demand", drawstyle="steps-post", marker="o", lw=2, color="tab:blue")
axes[0].set(ylabel="Demand (kWh)")

ax_02 = axes[0].twinx()
ax_02.plot(price_forecast[:simulation_timesteps], label="Price", drawstyle="steps-post", marker="x", lw=2, color="tab:orange")
ax_02.set(ylabel="Price (£ / kWh)")

# Create shared legend
handles_main, labels_main = axes[0].get_legend_handles_labels()
handles_twin, labels_twin = ax_02.get_legend_handles_labels()
axes[0].legend(handles_main + handles_twin, labels_main + labels_twin, loc=4)

axes[1].plot(grid_flow_history_greedy, label="Greedy", lw=2, drawstyle="steps-post")
axes[1].plot(grid_flow_history_mpc, label="MPC", lw=2, drawstyle="steps-post")
axes[1].set(ylabel="Grid Inflow (kWh)")
axes[1].legend(loc=1)

axes[2].plot(state_of_charge_history_greedy, label="Greedy", lw=2, drawstyle="steps-post")
axes[2].plot(state_of_charge_history_mpc, label="MPC", lw=2, drawstyle="steps-post")
axes[2].set(ylabel="State of Charge (kWh)")
axes[2].legend(loc=1)

axes[3].plot(np.cumsum(cost_history_greedy), label="Greedy", lw=2, drawstyle="steps-post")
axes[3].plot(np.cumsum(cost_history_mpc), label="MPC", lw=2, drawstyle="steps-post")
axes[3].set(ylabel="Cost (£)")
axes[3].legend(loc=4)

for ax in axes: ax.grid()

fig.align_labels();
fig.tight_layout();